In [11]:
# Phase 4
# Using ML methods to compare with BL Regression

from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix
from sklearn.preprocessing import StandardScaler

import pandas as pd
import numpy as np

In [7]:
train = pd.read_csv('../data/processed/race_driver_records_train_fe.csv')
test = pd.read_csv('../data/processed/race_driver_records_test_fe.csv')

selected_features = [
    'year', 'round', 'grid_position',
    'driver_podium_pct_y2', 'driver_avg_pos_y2', 'driver_podium_pct_y3',
    'constructor_races_y1', 'constructor_podium_pct_y2', 'constructor_podium_pct_y3',
    'driver_avg_pos_dry_3y', 'driver_avg_pos_wet_3y'
]

X_train = train[selected_features]
X_train = X_train.fillna(X_train.median())
y_train = train['is_podium']

X_test = test[selected_features]
X_test = X_test.fillna(X_test.median())
y_test = test['is_podium']

In [10]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Test: {X_test.shape[0]} samples, {X_test.shape[1]} features")
print(f"\nClass distribution (train): {y_train.value_counts().to_dict()}")
print(f"Class distribution (test): {y_test.value_counts().to_dict()}")

Train: 5857 samples, 11 features
Test: 240 samples, 11 features

Class distribution (train): {0: 5029, 1: 828}
Class distribution (test): {0: 204, 1: 36}


In [13]:
# Train Naive Bayes 

nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train)

y_train_pred_proba = nb_model.predict_proba(X_train_scaled)[:, 1]
fpr_train, tpr_train, thresholds_train = roc_curve(y_train, y_train_pred_proba)
train_auc = roc_auc_score(y_train, y_train_pred_proba)

j_scores = tpr_train - fpr_train
optimal_idx = np.argmax(j_scores)
optimal_threshold = thresholds_train[optimal_idx]


y_train_pred = (y_train_pred_proba >= optimal_threshold).astype(int)
cm_train = confusion_matrix(y_train, y_train_pred)

y_test_pred_proba = nb_model.predict_proba(X_test_scaled)[:, 1]
fpr_test, tpr_test, _ = roc_curve(y_test, y_test_pred_proba)
test_auc = roc_auc_score(y_test, y_test_pred_proba)

y_test_pred = (y_test_pred_proba >= optimal_threshold).astype(int)
cm_test = confusion_matrix(y_test, y_test_pred)

print("="*60)
print("NAIVE BAYES RESULTS")
print("="*60)
print(f"\nOptimal Threshold: {optimal_threshold:.4f}")
print(f"\nTrain AUC: {train_auc:.4f}")
print(f"Train Confusion Matrix:\n{cm_train}")
print(f"\nTest AUC: {test_auc:.4f}")
print(f"Test Confusion Matrix:\n{cm_test}")

NAIVE BAYES RESULTS

Optimal Threshold: 0.0354

Train AUC: 0.8844
Train Confusion Matrix:
[[4022 1007]
 [ 130  698]]

Test AUC: 0.8952
Test Confusion Matrix:
[[151  53]
 [  4  32]]


In [15]:
# Method 2: Decision Tree
dt_model = DecisionTreeClassifier(
    max_depth=10,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42
)
dt_model.fit(X_train, y_train)

y_train_pred_proba_dt = dt_model.predict_proba(X_train)[:, 1]
fpr_train_dt, tpr_train_dt, thresholds_train_dt = roc_curve(y_train, y_train_pred_proba_dt)
train_auc_dt = roc_auc_score(y_train, y_train_pred_proba_dt)

j_scores_dt = tpr_train_dt - fpr_train_dt
optimal_idx_dt = np.argmax(j_scores_dt)
optimal_threshold_dt = thresholds_train_dt[optimal_idx_dt]

y_train_pred_dt = (y_train_pred_proba_dt >= optimal_threshold_dt).astype(int)
cm_train_dt = confusion_matrix(y_train, y_train_pred_dt)

y_test_pred_proba_dt = dt_model.predict_proba(X_test)[:, 1]
fpr_test_dt, tpr_test_dt, _ = roc_curve(y_test, y_test_pred_proba_dt)
test_auc_dt = roc_auc_score(y_test, y_test_pred_proba_dt)

y_test_pred_dt = (y_test_pred_proba_dt >= optimal_threshold_dt).astype(int)
cm_test_dt = confusion_matrix(y_test, y_test_pred_dt)

print("="*60)
print("DECISION TREE RESULTS")
print("="*60)
print(f"\nOptimal Threshold: {optimal_threshold_dt:.4f}")
print(f"\nTrain AUC: {train_auc_dt:.4f}")
print(f"Train Confusion Matrix:\n{cm_train_dt}")
print(f"\nTest AUC: {test_auc_dt:.4f}")
print(f"Test Confusion Matrix:\n{cm_test_dt}")

DECISION TREE RESULTS

Optimal Threshold: 0.1538

Train AUC: 0.9676
Train Confusion Matrix:
[[4350  679]
 [  33  795]]

Test AUC: 0.9459
Test Confusion Matrix:
[[160  44]
 [  4  32]]


In [16]:
# Method 3: Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

y_train_pred_proba_rf = rf_model.predict_proba(X_train)[:, 1]
fpr_train_rf, tpr_train_rf, thresholds_train_rf = roc_curve(y_train, y_train_pred_proba_rf)
train_auc_rf = roc_auc_score(y_train, y_train_pred_proba_rf)

j_scores_rf = tpr_train_rf - fpr_train_rf
optimal_idx_rf = np.argmax(j_scores_rf)
optimal_threshold_rf = thresholds_train_rf[optimal_idx_rf]

y_train_pred_rf = (y_train_pred_proba_rf >= optimal_threshold_rf).astype(int)
cm_train_rf = confusion_matrix(y_train, y_train_pred_rf)

y_test_pred_proba_rf = rf_model.predict_proba(X_test)[:, 1]
fpr_test_rf, tpr_test_rf, _ = roc_curve(y_test, y_test_pred_proba_rf)
test_auc_rf = roc_auc_score(y_test, y_test_pred_proba_rf)

y_test_pred_rf = (y_test_pred_proba_rf >= optimal_threshold_rf).astype(int)
cm_test_rf = confusion_matrix(y_test, y_test_pred_rf)

print("="*60)
print("RANDOM FOREST RESULTS")
print("="*60)
print(f"\nOptimal Threshold: {optimal_threshold_rf:.4f}")
print(f"\nTrain AUC: {train_auc_rf:.4f}")
print(f"Train Confusion Matrix:\n{cm_train_rf}")
print(f"\nTest AUC: {test_auc_rf:.4f}")
print(f"Test Confusion Matrix:\n{cm_test_rf}")

RANDOM FOREST RESULTS

Optimal Threshold: 0.1909

Train AUC: 0.9735
Train Confusion Matrix:
[[4458  571]
 [  36  792]]

Test AUC: 0.9314
Test Confusion Matrix:
[[164  40]
 [  4  32]]
